# 03. ViT Self-Attention Rollout Visualization

This notebook implements **Attention Rollout** for Vision Transformers (ViT).

### Overview
1. Selects a target sample by `TARGET_STUDY_ID` (or picks a random test sample if empty).
2. Hooks into all self-attention layers $l = 1, \dots, L$ of the CLIP Vision Transformer backbone.
3. Captures multi-head self-attention weight matrices $A_l$.
4. Recursively propagates attention flow from the `[CLS]` token across layers:
   $$V = \prod_{l=1}^L \left( 0.5 \cdot A_l + 0.5 \cdot I \right)$$
5. Overlays the resulting spatial attention rollout heatmap with a **Warm Red vs. Transparent** colormap onto the 12-lead ECG image.

In [ ]:
import os
import sys
import yaml
import random
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from matplotlib.colors import LinearSegmentedColormap

# Ensure project root is in sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from filip.model.filip_ecg_model import FILIPECGModel
from filip.data.dataset import ECGImageDataset

# Custom Warm Red vs. Transparent Colormap (alpha 0.0 at 0, 0.75 warm red at peak)
colors = [(1.0, 0.0, 0.0, 0.0), (1.0, 0.15, 0.0, 0.75)]
warm_red_cmap = LinearSegmentedColormap.from_list('WarmRedTransparent', colors)

## 1. Load FILIP Vision Encoder Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Path to 23 PTB-XL Subclass adaptation config & checkpoint
# Default to newest ViT-Large report-aligned model checkpoint
checkpoint_dir = os.path.join(PROJECT_ROOT, 'outputs', 'filip', 'vit_large_ptbxl_sub_report_align_adapt_100')
if not os.path.exists(checkpoint_dir):
    checkpoint_dir = os.path.join(PROJECT_ROOT, 'outputs', 'filip', 'ptbxl_sub_report_align_adapt_100')
if not os.path.exists(checkpoint_dir):
    checkpoint_dir = os.path.join(PROJECT_ROOT, 'outputs', 'filip', 'ptbxl_sub_adapt_100')

checkpoint_path = os.path.join(checkpoint_dir, 'best.pt')
if not os.path.exists(checkpoint_path):
    checkpoint_path = os.path.join(checkpoint_dir, 'best_model.pt')

# Select matching config file corresponding to checkpoint architecture
if 'vit_large' in checkpoint_dir:
    config_path = os.path.join(PROJECT_ROOT, 'filip', 'configs', 'report_alignment_adapt_vit_large', 'ptbxl_sub_adapt.yaml')
elif 'report_align' in checkpoint_dir:
    config_path = os.path.join(PROJECT_ROOT, 'filip', 'configs', 'report_alignment_adapt', 'ptbxl_sub_adapt.yaml')
else:
    config_path = os.path.join(PROJECT_ROOT, 'filip', 'configs', 'ptbxl_sub_adapt.yaml')

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print(f"Loading config from: {config_path}")

model = FILIPECGModel(config).to(device)
if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device)
    sd = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(sd, strict=False)
    print(f"Loaded FILIP model from: {checkpoint_path}")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}. Using base initialized model.")

model.eval()

## 2. Select Sample (Specified Study ID or Random Selection)

In [ ]:
from filip.visualization.diagnostic_utils import ExpandToSquare
# Set TARGET_STUDY_ID to a specific Study ID (e.g. "12703_hr"), or leave empty "" for random selection
TARGET_STUDY_ID = ""

data_root = os.path.join(PROJECT_ROOT, config.get('data_root', 'data/ptbxl_sub_class/'))
image_size = config.get('model', {}).get('image_size', 224)
patch_size = config.get('model', {}).get('patch_size', 14)
grid_size = image_size // patch_size

transform = transforms.Compose([
    ExpandToSquare(background_color=(255, 255, 255)),
    transforms.Resize((image_size, image_size)),    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
])

test_dataset = ECGImageDataset(data_root=data_root, split='test', dataset_name=config.get('dataset_name', 'ptbxl_sub'), transform=transform)

# Select target record index
selected_idx = None
if TARGET_STUDY_ID.strip():
    for idx, record in enumerate(test_dataset.records):
        if str(record.get('study_id')) == TARGET_STUDY_ID.strip():
            selected_idx = idx
            break
    if selected_idx is None:
        print(f"Study ID '{TARGET_STUDY_ID}' not found in test set. Defaulting to random selection.")

if selected_idx is None:
    selected_idx = random.randint(0, len(test_dataset) - 1)

sample = test_dataset[selected_idx]
image_tensor = sample['images'].unsqueeze(0).to(device)
study_id = sample['sample_ids']
diagnosis_targets = sample['diagnosis_targets']

gt_labels = []
if diagnosis_targets is not None:
    for i, target_val in enumerate(diagnosis_targets):
        if target_val > 0.5:
            gt_labels.append(test_dataset.diagnosis_list[i])

print("=" * 60)
print(f"Selected Sample Index: {selected_idx} / {len(test_dataset)}")
print(f"Selected Study ID:     {study_id}")
print(f"Ground Truth Diagnoses: {gt_labels if gt_labels else 'None / Normal'}")
print("=" * 60)

## 3. Register Attention Hooks & Compute Rollout

In [ ]:
from filip.visualization.diagnostic_utils import ExpandToSquare
with torch.no_grad():
    encoder_outputs = model.vision_encoder.encoder(image_tensor, output_attentions=True)
    attentions = encoder_outputs.attentions

# Compute Attention Rollout on CPU to avoid device mismatch
num_tokens = attentions[0].size(-1)
result = torch.eye(num_tokens)

with torch.no_grad():
    for attn in attentions:
        attn_heads_avg = torch.mean(attn[0], dim=0).cpu() # [Tokens, Tokens] on CPU
        attn_with_residual = 0.5 * attn_heads_avg + 0.5 * torch.eye(num_tokens)
        attn_normalized = attn_with_residual / attn_with_residual.sum(dim=-1, keepdim=True)
        result = torch.matmul(attn_normalized, result)

cls_rollout = result[0, 1:].numpy().reshape(grid_size, grid_size)
cls_rollout = (cls_rollout - cls_rollout.min()) / (cls_rollout.max() - cls_rollout.min() + 1e-8)

print(f"Computed Attention Rollout Heatmap Shape: {cls_rollout.shape}")

## 4. Overlay Warm Red vs. Transparent Attention Rollout on ECG Image

In [ ]:
from filip.visualization.diagnostic_utils import ExpandToSquare
raw_img_path = os.path.join(data_root, 'images', f"{study_id}-0.png")
if not os.path.exists(raw_img_path):
    raw_img_path = os.path.join(data_root, 'images', f"{study_id}.png")

raw_image = Image.open(raw_img_path).convert('RGB').resize((image_size, image_size))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(raw_image)
axes[0].set_title(f"Original ECG (Study ID: {study_id})\nGT: {gt_labels if gt_labels else 'Normal'}")
axes[0].axis('off')

axes[1].imshow(raw_image)
im = axes[1].imshow(cls_rollout, cmap=warm_red_cmap, extent=(0, image_size, image_size, 0))
axes[1].set_title("Vision Transformer Self-Attention Rollout")
axes[1].axis('off')
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label="Rollout Attention")

plt.tight_layout()
plt.show()